# 关键对象
## 提示词模板

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage,HumanMessage, AIMessage
import os

load_dotenv()
model=init_chat_model(
    model="deepseek-chat"
)

    


c:\Users\Caihua-CSY\Desktop\LangChain-LN\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
# 定义消息
messages=[
    SystemMessage(content="你是我的人工智能助手"),
    HumanMessage(content="你是谁？"),
]



In [ ]:
# 阻塞式
response=model.invoke(messages)
print(response.content)

In [ ]:
# 流式
stream=model.stream(messages)
for chunk in stream:
    print(chunk.content,end="")

In [ ]:
from langchain_core.prompts import PromptTemplate

# 定义模板
template='你是一个{role},请用{style}风格回答问题:{question}'
# 创建模板对象
prompt_template=PromptTemplate.from_template(template)

# 变量的填充
# filled_prompt = prompt_template.format(role="数学老师",style="通俗易懂",question="黄金比例是什么？")
# 也可以使用invoke方法进行填充，使用字典进行填充。
filled_prompt = prompt_template.invoke({"role": "数学老师", "style": "通俗易懂", "question": "黄金比例是什么？"})
# 可以使用很多种方式进行填充，使用字典进行填充是最常见的方式。

In [ ]:
response = model.invoke(filled_prompt)
print(response.content)

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage,HumanMessage, AIMessage
import os
from langchain_core.prompts import PromptTemplate

# 定义模板
template='你是一个{role},请用{style}风格回答问题:{question}'
# 创建模板对象
prompt_template=PromptTemplate.from_template(template)

# 变量的填充
# filled_prompt = prompt_template.format(role="数学老师",style="通俗易懂",question="黄金比例是什么？")
# 也可以使用invoke方法进行填充，使用字典进行填充。
filled_prompt = prompt_template.invoke({"role": "数学老师", "style": "通俗易懂", "question": "黄金比例是什么？"})
# 可以使用很多种方式进行填充，使用字典进行填充是最常见的方式。

load_dotenv()
model=init_chat_model(
    model="deepseek-chat"
)
response = model.invoke(filled_prompt)
print(response.content)

## 输出格式化

In [22]:
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

# 定义你需要的输出结构
class MyOutput(BaseModel):
    name: str
    age: int
    description: str

# 初始化模型并开启结构化输出
structured_model = model.with_structured_output(MyOutput)

# 直接调用，返回的就是符合结构的对象
result = structured_model.invoke("帮我介绍一下张三，他今年30岁，是个软件工程师")
print(result)

name='张三' age=30 description='软件工程师'


In [23]:
from langchain_openai import ChatOpenAI
import json

# 1. 初始化模型，开启JSON原生模式
# 直接指定返回JSON格式的字典结构
json_model = model.with_structured_output(dict, method="json_mode")

# 2. 调用模型，直接得到可序列化的JSON对象
result = json_model.invoke("帮我生成一个用户信息，包含姓名、年龄、邮箱三个字段，用JSON返回")

# 3. 导出为标准JSON字符串
json_output = json.dumps(result, ensure_ascii=False, indent=2)
print(json_output)

{
  "姓名": "张明",
  "年龄": 28,
  "邮箱": "zhangming@example.com"
}


In [24]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama  # 以本地Llama3模型为例
import json

# 1. 初始化JSON解析器
parser = JsonOutputParser()

# 2. 定义Prompt，自动注入JSON格式指令
prompt = PromptTemplate(
    template="回答用户的问题，严格按照要求的格式输出。\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    # 自动生成格式说明，告诉模型要输出JSON
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 3. 构建处理链
chain = prompt | model | parser

# 4. 调用并导出JSON
result = chain.invoke({"query": "帮我生成一个产品信息，包含产品名、价格、分类三个字段"})
json_output = json.dumps(result, ensure_ascii=False, indent=2)
print(json_output)

{
  "产品名": "智能手表",
  "价格": 1299.0,
  "分类": "电子产品"
}
